### Understand Data

In [0]:
df_kna1 = spark.table("sap_sd_project.bronze.kna1_raw")
df_kna1.display()

MANDT,KUNNR,NAME1,LAND1,ORT01,REGIO,KTOKD,ERDAT
100,100001,Helwan Solutions Co,EG,Minya,MN,Z002,2022-02-12
100,100002,Red Sea Supplies Co,EG,Cairo,C,Z002,2023-11-27
100,100003,Sinai Manufacturing Co,EG,Damietta,null,Z001,2022-06-29
100,100004,Memphis Industries Co,EG,Cairo,C,Z001,2022-08-09
100,100005,Red Sea Distribution Co,EG,Zagazig,SH,Z001,2022-06-05
100,100006,Nile Distribution Co,EG,Alexandria,ALX,Z001,2022-12-27
100,100007,Pharaohs Solutions Co,EG,Zagazig,SH,Z001,2024-06-24
100,100008,Delta Logistics Co,EG,null,GZ,Z002,2022-06-28
100,100009,CANAL SUPPLIES CO,EG,null,ASN,Z002,2023-02-21
100,100010,Nile Logistics Co,EG,Mansoura,DK,Z001,2023-05-22


In [0]:
df_kna1.printSchema()

root
 |-- MANDT: long (nullable = true)
 |-- KUNNR: long (nullable = true)
 |-- NAME1: string (nullable = true)
 |-- LAND1: string (nullable = true)
 |-- ORT01: string (nullable = true)
 |-- REGIO: string (nullable = true)
 |-- KTOKD: string (nullable = true)
 |-- ERDAT: date (nullable = true)



In [0]:
df_kna1.describe()

DataFrame[summary: string, MANDT: string, KUNNR: string, NAME1: string, LAND1: string, ORT01: string, REGIO: string, KTOKD: string]

In [0]:
from pyspark.sql.functions import col,count,when
df_kna1.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df_kna1.columns
]).display()

MANDT,KUNNR,NAME1,LAND1,ORT01,REGIO,KTOKD,ERDAT
0,0,0,0,5,4,0,0


In [0]:
from pyspark.sql.functions import col, create_map, lit, coalesce, trim

city_region_map = {
    "Cairo": "C",
    "Giza": "GZ",
    "Alexandria": "ALX",
    "Mansoura": "DK",
    "Tanta": "GH",
    "Suez": "SUZ",
    "Ismailia": "IS",
    "Damietta": "DT",
    "Aswan": "ASN",
    "Minya": "MN",
    "Zagazig": "SH",
    "Port Said": "PTS"
}

region_city_map = {v: k for k, v in city_region_map.items()}

city_to_region = create_map(
    *[x for k, v in city_region_map.items() for x in (lit(k), lit(v))]
)

region_to_city = create_map(
    *[x for k, v in region_city_map.items() for x in (lit(k), lit(v))]
)

df_kna1_clean = (
    df_kna1
    .withColumn("ORT01", trim(col("ORT01")))
    .withColumn("REGIO", trim(col("REGIO")))

    .withColumn(
        "ORT01",
        coalesce(
            col("ORT01"),
            region_to_city[col("REGIO")]
        )
    )

    .withColumn(
        "REGIO",
        coalesce(
            col("REGIO"),
            city_to_region[col("ORT01")]
        )
    )
)

display(df_kna1_clean)

MANDT,KUNNR,NAME1,LAND1,ORT01,REGIO,KTOKD,ERDAT
100,100001,Helwan Solutions Co,EG,Minya,MN,Z002,2022-02-12
100,100002,Red Sea Supplies Co,EG,Cairo,C,Z002,2023-11-27
100,100003,Sinai Manufacturing Co,EG,Damietta,DT,Z001,2022-06-29
100,100004,Memphis Industries Co,EG,Cairo,C,Z001,2022-08-09
100,100005,Red Sea Distribution Co,EG,Zagazig,SH,Z001,2022-06-05
100,100006,Nile Distribution Co,EG,Alexandria,ALX,Z001,2022-12-27
100,100007,Pharaohs Solutions Co,EG,Zagazig,SH,Z001,2024-06-24
100,100008,Delta Logistics Co,EG,Giza,GZ,Z002,2022-06-28
100,100009,CANAL SUPPLIES CO,EG,Aswan,ASN,Z002,2023-02-21
100,100010,Nile Logistics Co,EG,Mansoura,DK,Z001,2023-05-22


In [0]:
df_kna1_clean.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in ["ORT01", "REGIO"]
]).display()

ORT01,REGIO
0,0


In [0]:
from pyspark.sql.functions import col, trim, initcap

df_kna1_clean = (
    df_kna1_clean
    .withColumn("NAME1", trim(col("NAME1")))
    .withColumn("NAME1", initcap(col("NAME1")))
)

In [0]:
display(
    df_kna1_clean.select("KUNNR", "NAME1").orderBy("NAME1")
)

KUNNR,NAME1
100054,Alex Distribution Co
100037,Alex Manufacturing Co
100085,Alex Retail Co
100098,Cairo Distribution Co
100018,Cairo Retail Co
100035,Cairo Supplies Co
100059,Cairo Supplies Co
100030,Cairo Trading Co
100072,Cairo Trading Co
100047,Canal Logistics Co


In [0]:
display(
    df_kna1_clean
    .groupBy("MANDT", "KUNNR")
    .count()
    .filter(col("count") > 1)
)

MANDT,KUNNR,count
100,100003,2
100,100005,2
100,100013,2
100,100045,2
100,100047,2


In [0]:
df_kna1_clean = df_kna1_clean.dropDuplicates(["MANDT", "KUNNR"])

In [0]:
print("Before:", df_kna1.count())
print("After :", df_kna1_clean.count())

Before: 105
After : 100


In [0]:
from pyspark.sql.functions import col, trim, upper, when, lpad

df_kna1_clean = (
    df_kna1_clean
    .withColumn("LAND1", upper(trim(col("LAND1"))))
    .withColumn(
        "LAND1",
        when(col("LAND1").isin("EG","EGYPT"),"EG")
        .otherwise(col("LAND1"))
    )
)

In [0]:
df_kna1_clean = (
    df_kna1_clean
    .withColumn("MANDT",col("MANDT").cast("string"))
    .withColumn(
        "KUNNR",
        lpad(col("KUNNR").cast("string"),10,"0")
    )
)

In [0]:
df_kna1_clean.printSchema()
display(df_kna1_clean)

root
 |-- MANDT: string (nullable = true)
 |-- KUNNR: string (nullable = true)
 |-- NAME1: string (nullable = true)
 |-- LAND1: string (nullable = true)
 |-- ORT01: string (nullable = true)
 |-- REGIO: string (nullable = true)
 |-- KTOKD: string (nullable = true)
 |-- ERDAT: date (nullable = true)



MANDT,KUNNR,NAME1,LAND1,ORT01,REGIO,KTOKD,ERDAT
100,0000100001,Helwan Solutions Co,EG,Minya,MN,Z002,2022-02-12
100,0000100002,Red Sea Supplies Co,EG,Cairo,C,Z002,2023-11-27
100,0000100003,Sinai Manufacturing Co,EG,Damietta,DT,Z001,2022-06-29
100,0000100004,Memphis Industries Co,EG,Cairo,C,Z001,2022-08-09
100,0000100005,Red Sea Distribution Co,EG,Zagazig,SH,Z001,2022-06-05
100,0000100006,Nile Distribution Co,EG,Alexandria,ALX,Z001,2022-12-27
100,0000100007,Pharaohs Solutions Co,EG,Zagazig,SH,Z001,2024-06-24
100,0000100008,Delta Logistics Co,EG,Giza,GZ,Z002,2022-06-28
100,0000100009,Canal Supplies Co,EG,Aswan,ASN,Z002,2023-02-21
100,0000100010,Nile Logistics Co,EG,Mansoura,DK,Z001,2023-05-22


In [0]:
from pyspark.sql.functions import col, count, when

# 1) عدد الصفوف
print("Rows:", df_kna1_clean.count())

# 2) Nulls
df_kna1_clean.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df_kna1_clean.columns
]).display()

# 3) Duplicates على Customer Key
display(
    df_kna1_clean
    .groupBy("MANDT", "KUNNR")
    .count()
    .filter(col("count") > 1)
)

# 4) Check Country values
display(
    df_kna1_clean
    .select("LAND1")
    .distinct()
)

Rows: 100


MANDT,KUNNR,NAME1,LAND1,ORT01,REGIO,KTOKD,ERDAT
0,0,0,0,0,0,0,0


MANDT,KUNNR,count


LAND1
EG


In [0]:
df_kna1_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("sap_sd_project.silver.kna1_clean")

In [0]:
display(
    spark.table("sap_sd_project.silver.kna1_clean")
)

MANDT,KUNNR,NAME1,LAND1,ORT01,REGIO,KTOKD,ERDAT
100,0000100028,Canal Solutions Co,EG,Zagazig,SH,Z002,2022-06-16
100,0000100039,Red Sea Distribution Co,EG,Ismailia,IS,Z002,2022-07-07
100,0000100076,Upper Egypt Distribution Co,EG,Cairo,C,Z002,2023-03-30
100,0000100077,Red Sea Logistics Co,EG,Suez,SUZ,Z002,2024-04-30
100,0000100097,Pyramids Logistics Co,EG,Zagazig,SH,Z002,2024-08-09
100,0000100011,Red Sea Retail Co,EG,Minya,MN,Z001,2023-06-25
100,0000100014,Mansoura Foods Co,EG,Port Said,PTS,Z001,2023-07-24
100,0000100065,Helwan Foods Co,EG,Tanta,GH,Z001,2023-03-23
100,0000100066,Nile Logistics Co,EG,Tanta,GH,Z001,2022-04-02
100,0000100075,New Cairo Logistics Co,EG,Minya,MN,Z002,2023-06-13


In [0]:
spark.table("sap_sd_project.silver.kna1_clean").count()

100